# Trabajo Práctico 2 - Grupo 02

### Modelo XGBoost

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen


El objetivo es entrenar un modelo de XGBoost sobre dos representaciones de texto (TF-IDF y BoW) con búsqueda de hiperparámetros, comparando su desempeño contra el baseline de Naive Bayes (~ 0.66 F1-macro en Kaggle) y contra el de Random Forest (~ 0.65 F1-macro en Kaggle).

XGBoost (eXtreme Gradient Boosting) es una técnica de aprendizaje por ensambles (ensemble learning), específicamente de tipo boosting. Combina múltiples modelos predictivos básicos (conocidos como "aprendices débiles"), generalmente árboles de decisión, para crear un modelo final altamente preciso y robusto.

## Importación e instalación de dependencias

In [1]:
!pip install -q spacy
!python -m spacy download es_core_news_sm
!pip install xgboost

  Using cached es_core_news_sm-3.8.0-py3-none-any.whl (12.9 MB)
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight

In [3]:
import sys
sys.path.insert(0, "../../..")  # sube tres niveles: v3 → XGBoost → Entregas → TP2

In [4]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, make_bow, SEED
from common.evaluation import evaluate
from common.io_utils import save_model, load_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [5]:
train = pd.read_csv("../../../data/train.csv")
test  = pd.read_csv("../../../data/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [6]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N3: XGBoost + TF-IDF con RandomizedSearch (20 iter)

In [7]:
# Agrupa vectorizador y modelo en un solo objeto
pipe = Pipeline([
    ("tfidf", make_tfidf()),
    ("xgb",   XGBClassifier(random_state=SEED, n_jobs=-1, eval_metric="mlogloss"))])

param_dist = {
    "xgb__n_estimators":     [100, 200, 300],
    "xgb__max_depth":        [3, 5, 7],
    "xgb__learning_rate":    [0.05, 0.1, 0.2],
    "xgb__subsample":        [0.7, 0.8, 1.0],
    "xgb__colsample_bytree": [0.5, 0.7, 1.0],
    "xgb__min_child_weight": [1, 3, 5],
}

rs_xgb_tfidf = RandomizedSearchCV(
    pipe, param_dist,
    n_iter=20, cv=5, scoring="f1_macro",
    n_jobs=2, random_state=SEED, verbose=1
)

print("Buscando mejores hiperparámetros XGBoost + TF-IDF")
weights_train = compute_sample_weight("balanced", y_train)
rs_xgb_tfidf.fit(X_train, y_train, xgb__sample_weight=weights_train)

print(f"\nMejores hiperparámetros: {rs_xgb_tfidf.best_params_}")
print(f"Mejor F1-macro en CV: {rs_xgb_tfidf.best_score_:.4f}")

y_pred = rs_xgb_tfidf.best_estimator_.predict(X_val)
evaluate("xgb_tfidf_v3", y_val, y_pred, hyperparams=rs_xgb_tfidf.best_params_)

y_full = np.concatenate([y_train, y_val])
weights_full = compute_sample_weight("balanced", y_full)
best_pipe = rs_xgb_tfidf.best_estimator_
best_pipe.fit(np.concatenate([X_train, X_val]), y_full, xgb__sample_weight=weights_full)
save_model(best_pipe, "xgb_tfidf_v3")
make_submission(best_pipe, pd.DataFrame({"id": test["id"], "text": X_test}), "submission_xgb_tfidf_v3.csv");

Buscando mejores hiperparámetros XGBoost + TF-IDF
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Mejores hiperparámetros: {'xgb__subsample': 0.8, 'xgb__n_estimators': 300, 'xgb__min_child_weight': 1, 'xgb__max_depth': 7, 'xgb__learning_rate': 0.2, 'xgb__colsample_bytree': 1.0}
Mejor F1-macro en CV: 0.6534

=== xgb_tfidf_v3 ===
Hiperparámetros: {'xgb__subsample': 0.8, 'xgb__n_estimators': 300, 'xgb__min_child_weight': 1, 'xgb__max_depth': 7, 'xgb__learning_rate': 0.2, 'xgb__colsample_bytree': 1.0}

F1-macro:  0.6619
Precision: 0.6641
Recall:    0.6639
Accuracy:  0.6988

              precision    recall  f1-score   support

    negativa     0.7785    0.7419    0.7598      4080
      neutra     0.3976    0.4892    0.4387      2040
    positiva     0.8161    0.7605    0.7874      4080

    accuracy                         0.6988     10200
   macro avg     0.6641    0.6639    0.6619     10200
weighted avg     0.7174    0.6988    0.7066     10200

Matriz de confusión (filas=